サイズ1→64

襞領域1→64

形状3→64

CNN64

を合成する

多分意味ないのでやらない

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"4AFFmodel_{TARGET_EPOCH}epoch"


# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. ResBlock + CBAM ---
class ResBlockWithCBAM(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlockWithCBAM, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.cbam = CBAM(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv_block(x)
        out = self.cbam(out)
        return torch.relu(out + identity)

# --- 3. 融合モデル (★ アプローチ1: 階層的融合法に変更) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # --- 特徴抽出・投影 ---
        self.dl_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResBlockWithCBAM(32, 64),
            ResBlockWithCBAM(64, 64)
        )
        
        self.proj_size = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_R = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_hu = nn.Sequential(nn.Linear(3, 64), nn.ReLU(inplace=True))
        
        # --- ★ 階層的融合用の3つのCBAMモジュール ---
        # Level 1-A 用 (CNN + Size)
        self.aff_1 = CBAM(64)
        # Level 1-B 用 (Hu + R)
        self.aff_2 = CBAM(64)
        # Level 2 (Final) 用 (Result_A + Result_B)
        self.aff_final = CBAM(64)
        
        # --- 分類器 ---
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    # [cite_start]★ 論文数式(1)に基づく融合メソッド [cite: 111]
    # Z = M(X + Y) * X + (1 - M(X + Y)) * Y
    def aff_fusion(self, x, y, attention_module):
        summed = x + y
        weights = attention_module(summed) # これが M(X+Y)
        return weights * x + (1 - weights) * y

    def forward(self, img, hc_vec):
        # 1. 各特徴量の準備 (すべて B, 64, H, W に揃える)
        feat_cnn = self.dl_extractor(img)
        B, C, H, W = feat_cnn.shape
        
        # Size (index 8)
        feat_size = self.proj_size(hc_vec[:, 8:9]).view(B, C, 1, 1).expand(B, C, H, W)
        # R (index 9)
        feat_R = self.proj_R(hc_vec[:, 9:10]).view(B, C, 1, 1).expand(B, C, H, W)
        # Hu (index 2,3,7)
        feat_hu = self.proj_hu(hc_vec[:, [2, 3, 7]]).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 2. 階層的融合 (Hierarchical Fusion)
        
        # Level 1: ペアごとの融合
        # ペアA: CNN と Size
        fused_A = self.aff_fusion(feat_cnn, feat_size, self.aff_1)
        
        # ペアB: Hu と R
        fused_B = self.aff_fusion(feat_hu, feat_R, self.aff_2)
        
        # Level 2: 最終融合
        # A と B を融合
        final_feat = self.aff_fusion(fused_A, fused_B, self.aff_final)
        
        # 3. 分類
        return self.classifier(final_feat)

# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 5. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # --- Lossのプロット ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)  # ★ Lossの軸を 0.0 ~ 1.2 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    # --- Accuracyのプロット ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)  # ★ Accuracyの軸を 0.0 ~ 1.0 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (階層的融合モデル)
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.2397 | Val Acc: 0.3889
  --> Best Model Saved (Acc: 0.3889)
Epoch 2/100 | Loss: 1.0843 | Val Acc: 0.4583
  --> Best Model Saved (Acc: 0.4583)
Epoch 3/100 | Loss: 1.0762 | Val Acc: 0.4653
  --> Best Model Saved (Acc: 0.4653)
Epoch 4/100 | Loss: 1.0497 | Val Acc: 0.4931
  --> Best Model Saved (Acc: 0.4931)
Epoch 5/100 | Loss: 1.0356 | Val Acc: 0.4931
Learning curve saved to 4AFFmodel_100epoch/learning_curve.png
Epoch 6/100 | Loss: 1.0137 | Val Acc: 0.5208
  --> Best Model Saved (Acc: 0.5208)
Epoch 7/100 | Loss: 0.9949 | Val Acc: 0.5278
  --> Best Model Saved (Acc: 0.5278)
Epoch 8/100 | Loss: 0.9721 | Val Acc: 0.6042
  --> Best Model Saved (Acc: 0.6042)
Epoch 9/100 | Loss: 0.9685 | Val Acc: 0.5347
Epoch 10/100 | Loss: 0.9595 | Val Acc: 0.5208
Learning curve saved to 4AFFmodel_100epoch/learning_curve.png
Epoch 11/100 | Loss: 0.9265 | Val Acc: 0.5972
Epoch 12/

CNNを小さいモデルに変更VGGネット→Resnet-10Lite

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"4AFF_Res10Lite_model_{TARGET_EPOCH}epoch"


# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. Small ResNet用 BasicBlock + CBAM ---
class BasicBlockWithCBAM(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlockWithCBAM, self).__init__()
        
        # 1つ目の畳み込み
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # 2つ目の畳み込み
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # CBAMモジュール
        self.cbam = CBAM(out_channels)

        # ショートカット接続
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        
        # CBAM適用
        out = self.cbam(out)

        # 残差結合
        out += identity
        out = self.relu(out)

        return out

# --- 3. Small ResNet (ResNet-10 Lite) ---
class SmallResNet(nn.Module):
    def __init__(self):
        super(SmallResNet, self).__init__()
        
        # 初期層: 256 -> 128
        self.in_channels = 16
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        # ResNetステージ: [16ch -> 32ch -> 64ch -> 64ch]
        # stride=2 でダウンサンプリング
        self.layer1 = self._make_layer(16, blocks=1, stride=1)
        self.layer2 = self._make_layer(32, blocks=1, stride=2) # 128 -> 64
        self.layer3 = self._make_layer(64, blocks=1, stride=2) # 64 -> 32
        self.layer4 = self._make_layer(64, blocks=1, stride=1) # 32 -> 32 (チャネル維持)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlockWithCBAM(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(BasicBlockWithCBAM(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

# --- 4. 融合モデル (階層的融合法 + SmallResNet) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # 軽量CNN特徴抽出器
        self.dl_extractor = SmallResNet()
        
        # ハンドクラフト特徴の投影層 (1次元or3次元 -> 64次元)
        self.proj_size = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_R = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_hu = nn.Sequential(nn.Linear(3, 64), nn.ReLU(inplace=True))
        
        # 階層的融合用のCBAMモジュール
        # Level 1-A (CNN + Size)
        self.aff_1 = CBAM(64)
        # Level 1-B (Hu + R)
        self.aff_2 = CBAM(64)
        # Level 2 (Final)
        self.aff_final = CBAM(64)
        
        # 分類器
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    # [cite_start]論文数式(1)に基づく融合メソッド [cite: 111]
    # Z = M(X + Y) * X + (1 - M(X + Y)) * Y
    def aff_fusion(self, x, y, attention_module):
        summed = x + y
        weights = attention_module(summed) 
        return weights * x + (1 - weights) * y

    def forward(self, img, hc_vec):
        # 1. CNN特徴抽出
        feat_cnn = self.dl_extractor(img) # (B, 64, H/8, W/8)
        B, C, H, W = feat_cnn.shape
        
        # 2. ハンドクラフト特徴の投影と拡張
        # Size (index 8)
        feat_size = self.proj_size(hc_vec[:, 8:9]).view(B, C, 1, 1).expand(B, C, H, W)
        # R (index 9)
        feat_R = self.proj_R(hc_vec[:, 9:10]).view(B, C, 1, 1).expand(B, C, H, W)
        # Hu (index 2,3,7)
        feat_hu = self.proj_hu(hc_vec[:, [2, 3, 7]]).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 3. 階層的融合 (Hierarchical Fusion)
        # Level 1
        fused_A = self.aff_fusion(feat_cnn, feat_size, self.aff_1)
        fused_B = self.aff_fusion(feat_hu, feat_R, self.aff_2)
        
        # Level 2
        final_feat = self.aff_fusion(fused_A, fused_B, self.aff_final)
        
        # 4. 分類
        return self.classifier(final_feat)

# --- 5. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 6. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # --- Lossのプロット ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)  # ★ Lossの軸を 0.0 ~ 1.2 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    # --- Accuracyのプロット ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)  # ★ Accuracyの軸を 0.0 ~ 1.0 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 7. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (階層的融合モデル + SmallResNet)
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.0655 | Val Acc: 0.5972
  --> Best Model Saved (Acc: 0.5972)
Epoch 2/100 | Loss: 0.9487 | Val Acc: 0.6042
  --> Best Model Saved (Acc: 0.6042)
Epoch 3/100 | Loss: 0.8829 | Val Acc: 0.5972
Epoch 4/100 | Loss: 0.8372 | Val Acc: 0.5903
Epoch 5/100 | Loss: 0.7840 | Val Acc: 0.5903
Learning curve saved to 4AFF_Res10Lite_model_100epoch/learning_curve.png
Epoch 6/100 | Loss: 0.7335 | Val Acc: 0.5833
Epoch 7/100 | Loss: 0.7069 | Val Acc: 0.6250
  --> Best Model Saved (Acc: 0.6250)
Epoch 8/100 | Loss: 0.6262 | Val Acc: 0.5417
Epoch 9/100 | Loss: 0.6043 | Val Acc: 0.5069
Epoch 10/100 | Loss: 0.5806 | Val Acc: 0.5694
Learning curve saved to 4AFF_Res10Lite_model_100epoch/learning_curve.png
Epoch 11/100 | Loss: 0.5275 | Val Acc: 0.5417
Epoch 12/100 | Loss: 0.4297 | Val Acc: 0.5208
Epoch 13/100 | Loss: 0.4133 | Val Acc: 0.5833
Epoch 14/100 | Loss: 0.3599 | Val Acc: 0.44

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"4AFF_ResNet18_model_{TARGET_EPOCH}epoch"


# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. ResNet用 BasicBlock + CBAM ---
class BasicBlockWithCBAM(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlockWithCBAM, self).__init__()
        
        # 1つ目の畳み込み
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # 2つ目の畳み込み
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # CBAMモジュール
        self.cbam = CBAM(out_channels)

        # ショートカット接続
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        
        # CBAM適用
        out = self.cbam(out)

        # 残差結合
        out += identity
        out = self.relu(out)

        return out

# --- 3. ResNet-18 (標準構造 + 出力圧縮) ---
class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        
        # Stem (初期層): 7x7 Conv, Stride 2 -> MaxPool
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet18 Stages: [2, 2, 2, 2] blocks
        # Channels: 64 -> 128 -> 256 -> 512
        self.layer1 = self._make_layer(64,  blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=2)
        
        # 次元圧縮層: 512次元 -> 64次元
        # FusionModelが64次元での結合を前提としているため
        self.compress = nn.Sequential(
            nn.Conv2d(512, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlockWithCBAM(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(BasicBlockWithCBAM(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Compression
        x = self.compress(x)
        return x

# --- 4. 融合モデル (階層的融合法 + ResNet18) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # CNN特徴抽出器 (ResNet18に変更)
        self.dl_extractor = ResNet18()
        
        # ハンドクラフト特徴の投影層 (1次元or3次元 -> 64次元)
        self.proj_size = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_R = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_hu = nn.Sequential(nn.Linear(3, 64), nn.ReLU(inplace=True))
        
        # 階層的融合用のCBAMモジュール
        # Level 1-A (CNN + Size)
        self.aff_1 = CBAM(64)
        # Level 1-B (Hu + R)
        self.aff_2 = CBAM(64)
        # Level 2 (Final)
        self.aff_final = CBAM(64)
        
        # 分類器
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    # 論文数式(1)に基づく融合メソッド
    def aff_fusion(self, x, y, attention_module):
        summed = x + y
        weights = attention_module(summed) 
        return weights * x + (1 - weights) * y

    def forward(self, img, hc_vec):
        # 1. CNN特徴抽出
        feat_cnn = self.dl_extractor(img) # (B, 64, H/32, W/32) -> ResNet18 output is 1/32 size
        B, C, H, W = feat_cnn.shape
        
        # 2. ハンドクラフト特徴の投影と拡張
        # Size (index 8)
        feat_size = self.proj_size(hc_vec[:, 8:9]).view(B, C, 1, 1).expand(B, C, H, W)
        # R (index 9)
        feat_R = self.proj_R(hc_vec[:, 9:10]).view(B, C, 1, 1).expand(B, C, H, W)
        # Hu (index 2,3,7)
        feat_hu = self.proj_hu(hc_vec[:, [2, 3, 7]]).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 3. 階層的融合 (Hierarchical Fusion)
        # Level 1
        fused_A = self.aff_fusion(feat_cnn, feat_size, self.aff_1)
        fused_B = self.aff_fusion(feat_hu, feat_R, self.aff_2)
        
        # Level 2
        final_feat = self.aff_fusion(fused_A, fused_B, self.aff_final)
        
        # 4. 分類
        return self.classifier(final_feat)

# --- 5. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 6. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # --- Lossのプロット ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)  # ★ Lossの軸を 0.0 ~ 1.2 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    # --- Accuracyのプロット ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)  # ★ Accuracyの軸を 0.0 ~ 1.0 に固定
    plt.legend()        # labelを表示するために必要
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 7. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (階層的融合モデル + ResNet18)
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.0958 | Val Acc: 0.5139
  --> Best Model Saved (Acc: 0.5139)
Epoch 2/100 | Loss: 1.0186 | Val Acc: 0.4306
Epoch 3/100 | Loss: 1.0148 | Val Acc: 0.3889
Epoch 4/100 | Loss: 1.0078 | Val Acc: 0.5556
  --> Best Model Saved (Acc: 0.5556)
Epoch 5/100 | Loss: 0.9890 | Val Acc: 0.5625
  --> Best Model Saved (Acc: 0.5625)
Learning curve saved to 4AFF_ResNet18_model_100epoch/learning_curve.png
Epoch 6/100 | Loss: 0.9503 | Val Acc: 0.5903
  --> Best Model Saved (Acc: 0.5903)
Epoch 7/100 | Loss: 0.9026 | Val Acc: 0.5694
Epoch 8/100 | Loss: 0.8997 | Val Acc: 0.5833
Epoch 9/100 | Loss: 0.8746 | Val Acc: 0.5972
  --> Best Model Saved (Acc: 0.5972)
Epoch 10/100 | Loss: 0.8389 | Val Acc: 0.5694
Learning curve saved to 4AFF_ResNet18_model_100epoch/learning_curve.png
Epoch 11/100 | Loss: 0.8393 | Val Acc: 0.5972
Epoch 12/100 | Loss: 0.7894 | Val Acc: 0.5347
Epoch 13/100 | Lo

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"4AFF_ConvNeXt_model_{TARGET_EPOCH}epoch"


# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. ConvNeXt パーツ定義 ---

class LayerNorm2d(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.normalized_shape = (normalized_shape, )

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0.):
        super().__init__()
        # 1. Depthwise Conv 7x7
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim) 
        self.norm = LayerNorm2d(dim, eps=1e-6)
        
        # 2. Pointwise Conv (Expansion)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        
        # 3. Pointwise Conv (Projection)
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = self.norm(x)
        
        # Conv2d (NCHW) -> Linear (NHWC) に変換して計算
        x = x.permute(0, 2, 3, 1) 
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2) # 元に戻す

        x = input + x # 残差結合
        return x

# --- 3. ConvNeXt (Custom Tiny) ---
class ConvNeXt(nn.Module):
    def __init__(self):
        super(ConvNeXt, self).__init__()
        
        # 設定: ConvNeXt-Tinyより少し軽量化 (チャンネル数を減らしています)
        # standard tiny: dims=[96, 192, 384, 768], depths=[3, 3, 9, 3]
        # custom: dims=[64, 128, 256, 512], depths=[2, 2, 6, 2]
        
        self.depths = [2, 2, 6, 2]
        self.dims = [64, 128, 256, 512]
        
        # Stem: 4x4 conv, stride 4 (画像サイズを一気に1/4にする)
        self.downsample_layers = nn.ModuleList() 
        stem = nn.Sequential(
            nn.Conv2d(3, self.dims[0], kernel_size=4, stride=4),
            LayerNorm2d(self.dims[0], eps=1e-6)
        )
        self.downsample_layers.append(stem)
        
        # Downsample layers for stages 2-4
        for i in range(3):
            downsample_layer = nn.Sequential(
                LayerNorm2d(self.dims[i], eps=1e-6),
                nn.Conv2d(self.dims[i], self.dims[i+1], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(downsample_layer)

        # Stages
        self.stages = nn.ModuleList() 
        for i in range(4):
            stage = nn.Sequential(
                *[ConvNeXtBlock(dim=self.dims[i]) for _ in range(self.depths[i])]
            )
            self.stages.append(stage)
        
        # 次元圧縮層: 512次元 -> 64次元
        self.compress = nn.Sequential(
            nn.Conv2d(self.dims[-1], 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        
        # 最終出力: (B, 512, H/32, W/32) -> (B, 64, H/32, W/32)
        x = self.compress(x)
        return x

# --- 4. 融合モデル (階層的融合法 + ConvNeXt) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # CNN特徴抽出器 (ConvNeXtに変更)
        self.dl_extractor = ConvNeXt()
        
        # ハンドクラフト特徴の投影層 (1次元or3次元 -> 64次元)
        self.proj_size = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_R = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_hu = nn.Sequential(nn.Linear(3, 64), nn.ReLU(inplace=True))
        
        # 階層的融合用のCBAMモジュール
        self.aff_1 = CBAM(64)
        self.aff_2 = CBAM(64)
        self.aff_final = CBAM(64)
        
        # 分類器
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    # 論文数式(1)に基づく融合メソッド
    def aff_fusion(self, x, y, attention_module):
        summed = x + y
        weights = attention_module(summed) 
        return weights * x + (1 - weights) * y

    def forward(self, img, hc_vec):
        # 1. CNN特徴抽出
        feat_cnn = self.dl_extractor(img) # (B, 64, H/32, W/32)
        B, C, H, W = feat_cnn.shape
        
        # 2. ハンドクラフト特徴の投影と拡張
        feat_size = self.proj_size(hc_vec[:, 8:9]).view(B, C, 1, 1).expand(B, C, H, W)
        feat_R = self.proj_R(hc_vec[:, 9:10]).view(B, C, 1, 1).expand(B, C, H, W)
        feat_hu = self.proj_hu(hc_vec[:, [2, 3, 7]]).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 3. 階層的融合 (Hierarchical Fusion)
        fused_A = self.aff_fusion(feat_cnn, feat_size, self.aff_1)
        fused_B = self.aff_fusion(feat_hu, feat_R, self.aff_2)
        
        final_feat = self.aff_fusion(fused_A, fused_B, self.aff_final)
        
        # 4. 分類
        return self.classifier(final_feat)

# --- 5. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 6. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)
    plt.legend()
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 7. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    # 画像サイズ調整 (ConvNeXtはStemで1/4にするため224x224以上が望ましい)
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (階層的融合モデル + ConvNeXt)
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    # ConvNeXtはAdamWが推奨されることが多いですが、ここでは統一のためAdamを使用
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.0750 | Val Acc: 0.4514
  --> Best Model Saved (Acc: 0.4514)
Epoch 2/100 | Loss: 1.0383 | Val Acc: 0.5556
  --> Best Model Saved (Acc: 0.5556)
Epoch 3/100 | Loss: 1.0083 | Val Acc: 0.6042
  --> Best Model Saved (Acc: 0.6042)
Epoch 4/100 | Loss: 0.9767 | Val Acc: 0.6181
  --> Best Model Saved (Acc: 0.6181)
Epoch 5/100 | Loss: 0.9544 | Val Acc: 0.6597
  --> Best Model Saved (Acc: 0.6597)
Learning curve saved to 4AFF_ConvNeXt_model_100epoch/learning_curve.png
Epoch 6/100 | Loss: 0.9291 | Val Acc: 0.6528
Epoch 7/100 | Loss: 0.9144 | Val Acc: 0.6389
Epoch 8/100 | Loss: 0.9011 | Val Acc: 0.6389
Epoch 9/100 | Loss: 0.8928 | Val Acc: 0.6736
  --> Best Model Saved (Acc: 0.6736)
Epoch 10/100 | Loss: 0.8838 | Val Acc: 0.6319
Learning curve saved to 4AFF_ConvNeXt_model_100epoch/learning_curve.png
Epoch 11/100 | Loss: 0.8764 | Val Acc: 0.6806
  --> Best Model Saved (Ac

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# ★ 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"4AFF_EfficientNetB0_model_{TARGET_EPOCH}epoch"


# --- 1. CBAM モジュール (Fusion用) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out

# --- 2. EfficientNet用パーツ (MBConv, SEBlock) ---

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class SEBlock(nn.Module):
    """ Squeeze-and-Excitation Block """
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_dim, 1),
            Swish(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)

class MBConvBlock(nn.Module):
    """ Mobile Inverted Residual Bottleneck Block """
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        self.stride = stride
        self.use_residual = (self.stride == 1 and in_channels == out_channels)
        
        hidden_dim = in_channels * expand_ratio
        reduced_dim = max(1, int(in_channels * se_ratio))

        layers = []
        # 1. Expansion (1x1 Conv)
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                Swish()
            ])
        
        # 2. Depthwise Conv
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size, stride, padding=kernel_size//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            Swish()
        ])
        
        # 3. Squeeze-and-Excitation
        layers.append(SEBlock(hidden_dim, reduced_dim))
        
        # 4. Pointwise Conv (Projection)
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        else:
            return self.conv(x)

# --- 3. EfficientNet-B0 (Custom Implementation) ---
class EfficientNetB0(nn.Module):
    def __init__(self):
        super(EfficientNetB0, self).__init__()
        
        # Config: [expand_ratio, channels, repeats, stride, kernel_size]
        # EfficientNet-B0 configuration
        self.mb_config = [
            [1, 16, 1, 1, 3],
            [6, 24, 2, 2, 3],
            [6, 40, 2, 2, 5],
            [6, 80, 3, 2, 3],
            [6, 112, 3, 1, 5],
            [6, 192, 4, 2, 5],
            [6, 320, 1, 1, 3]
        ]
        
        # Stem
        self.in_channels = 32
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            Swish()
        )
        
        # Build Blocks
        layers = []
        for t, c, n, s, k in self.mb_config:
            # 最初のブロックだけstrideを適用
            layers.append(MBConvBlock(self.in_channels, c, k, s, expand_ratio=t))
            self.in_channels = c
            # 残りの繰り返しブロック
            for _ in range(n - 1):
                layers.append(MBConvBlock(self.in_channels, c, k, 1, expand_ratio=t))
        
        self.blocks = nn.Sequential(*layers)
        
        # Head (Conv 1x1 to 1280)
        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            Swish()
        )
        
        # 次元圧縮層: 1280 -> 64 (Fusion用)
        self.compress = nn.Sequential(
            nn.Conv2d(1280, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.compress(x)
        return x

# --- 4. 融合モデル (階層的融合法 + EfficientNetB0) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # CNN特徴抽出器 (EfficientNetB0に変更)
        self.dl_extractor = EfficientNetB0()
        
        # ハンドクラフト特徴の投影層 (1次元or3次元 -> 64次元)
        self.proj_size = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_R = nn.Sequential(nn.Linear(1, 64), nn.ReLU(inplace=True))
        self.proj_hu = nn.Sequential(nn.Linear(3, 64), nn.ReLU(inplace=True))
        
        # 階層的融合用のCBAMモジュール
        # Level 1-A (CNN + Size)
        self.aff_1 = CBAM(64)
        # Level 1-B (Hu + R)
        self.aff_2 = CBAM(64)
        # Level 2 (Final)
        self.aff_final = CBAM(64)
        
        # 分類器
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    # 論文数式(1)に基づく融合メソッド
    def aff_fusion(self, x, y, attention_module):
        summed = x + y
        weights = attention_module(summed) 
        return weights * x + (1 - weights) * y

    def forward(self, img, hc_vec):
        # 1. CNN特徴抽出
        feat_cnn = self.dl_extractor(img) # (B, 64, H/32, W/32)
        B, C, H, W = feat_cnn.shape
        
        # 2. ハンドクラフト特徴の投影と拡張
        # Size (index 8)
        feat_size = self.proj_size(hc_vec[:, 8:9]).view(B, C, 1, 1).expand(B, C, H, W)
        # R (index 9)
        feat_R = self.proj_R(hc_vec[:, 9:10]).view(B, C, 1, 1).expand(B, C, H, W)
        # Hu (index 2,3,7)
        feat_hu = self.proj_hu(hc_vec[:, [2, 3, 7]]).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 3. 階層的融合 (Hierarchical Fusion)
        # Level 1
        fused_A = self.aff_fusion(feat_cnn, feat_size, self.aff_1)
        fused_B = self.aff_fusion(feat_hu, feat_R, self.aff_2)
        
        # Level 2
        final_feat = self.aff_fusion(fused_A, fused_B, self.aff_final)
        
        # 4. 分類
        return self.classifier(final_feat)

# --- 5. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 6. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0.0, 1.2)
    plt.legend()
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")

# --- 7. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    # EfficientNetは入力サイズが柔軟ですが、224x224以上が標準的
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル準備 (階層的融合モデル + EfficientNetB0)
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
Train samples: 576
Val samples:   144
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.1514 | Val Acc: 0.3333
  --> Best Model Saved (Acc: 0.3333)
Epoch 2/100 | Loss: 1.0633 | Val Acc: 0.3958
  --> Best Model Saved (Acc: 0.3958)
Epoch 3/100 | Loss: 0.9347 | Val Acc: 0.4028
  --> Best Model Saved (Acc: 0.4028)
Epoch 4/100 | Loss: 0.8686 | Val Acc: 0.3750
Epoch 5/100 | Loss: 0.6631 | Val Acc: 0.4167
  --> Best Model Saved (Acc: 0.4167)
Learning curve saved to 4AFF_EfficientNetB0_model_100epoch/learning_curve.png
Epoch 6/100 | Loss: 0.5849 | Val Acc: 0.4722
  --> Best Model Saved (Acc: 0.4722)
Epoch 7/100 | Loss: 0.3961 | Val Acc: 0.4583
Epoch 8/100 | Loss: 0.3218 | Val Acc: 0.4653
Epoch 9/100 | Loss: 0.2572 | Val Acc: 0.4653
Epoch 10/100 | Loss: 0.2873 | Val Acc: 0.4444
Learning curve saved to 4AFF_EfficientNetB0_model_100epoch/learning_curve.png
Epoch 11/100 | Loss: 0.1455 | Val Acc: 0.4167
Epoch 12/100 | Loss: 0.2425 | Val Acc: 0.4167
Epoch